In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import re

import pandas as pd
import numpy as np
import bambi as bmb
import arviz as az
from scipy.special import logit as logit_func

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import sse_detection.lib.sse_detection as sse_detection  # noqa: E402

In [2]:
cluster_data = pd.read_parquet("results/sse_outputs/cluster_table.parquet")
sequence_data = sse_detection.load_sequence_data()

In [3]:
priority_tiers = {
    "high_priority_both_axes",
    "high_priority_burst",
    "high_priority_burden",
}

high_priority = cluster_data["candidate_tier"].isin(priority_tiers)
cluster_data["candidate"] = high_priority.astype(bool)

candidate_sizes = cluster_data.loc[cluster_data["candidate"], "cluster_size"]

min_candidate_size = int(candidate_sizes.min())
eligible_nodes = cluster_data.loc[
    cluster_data["cluster_size"].ge(min_candidate_size)
].copy()

eligible_sequence_data = sequence_data.merge(
    eligible_nodes[["cluster_id", "candidate"]],
    on="cluster_id",
    how="inner",
)

candidate_sequence_rate = eligible_sequence_data["candidate"].mean()
candidate_node_rate = eligible_nodes["candidate"].mean()

## Bayesian logistic regression helpers

In [4]:
def sample_model_test_data(
    data: pd.DataFrame,
    *,
    outcome="candidate",
    max_rows=2000,
    positive_fraction=0.35,
    random_state=123,
    categorical_vars=None,
) -> pd.DataFrame:

    positives = data[data[outcome] == 1]
    negatives = data[data[outcome] == 0]

    n_pos = min(len(positives), int(max_rows * positive_fraction))
    n_neg = min(len(negatives), max_rows - n_pos)

    pos_sample = positives.sample(n=n_pos, random_state=random_state, replace=False)

    neg_sample = negatives.sample(n=n_neg, random_state=random_state, replace=False)

    sampled = pd.concat([pos_sample, neg_sample], axis=0)

    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)

    if categorical_vars is not None:
        for col in categorical_vars:
            if col in sampled.columns:
                sampled[col] = pd.Categorical(
                    sampled[col],
                    categories=sampled[col].dropna().unique(),
                )

    return sampled


def get_complete_case_data(
    df: pd.DataFrame,
    *,
    outcome="candidate",
    predictors=None,
    group_vars=("window_idx", "clade"),
    categorical_vars=("window_idx", "clade"),
    verbose=True,
) -> pd.DataFrame:
    """
    Create complete-case data for Bayesian logistic regression.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    outcome : str
        Binary outcome column.
    predictors : list[str]
        Predictor columns to include.
    group_vars : tuple/list[str]
        Grouping variables for random effects.
    categorical_vars : tuple/list[str]
        Columns to convert to categorical dtype.
    verbose : bool
        Print missingness and event-rate summary.

    Returns
    -------
    out : pandas.DataFrame
        Complete-case dataframe containing outcome, predictors, and group vars.
    """

    if predictors is None:
        raise ValueError("Please provide a list of predictor columns.")

    required_cols = [outcome, "cluster_id"] + list(predictors) + list(group_vars)

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in dataframe: {missing_cols}")

    before_n = len(df)

    out = df[required_cols].dropna().copy()

    after_n = len(out)
    dropped_n = before_n - after_n

    # Outcome should be integer 0/1
    out[outcome] = out[outcome].astype(int)

    # Convert grouping/categorical variables
    for col in categorical_vars:
        if col in out.columns:
            out[col] = out[col].astype("category")

    if verbose:
        print("Complete-case summary")
        print("---------------------")
        print(f"Rows before dropna: {before_n:,}")
        print(f"Rows after dropna:  {after_n:,}")
        print(f"Rows dropped:       {dropped_n:,}")
        print(f"Percent retained:   {after_n / before_n:.1%}")

        print("\nOutcome counts:")
        print(out[outcome].value_counts(dropna=False))

        print("\nOutcome proportions:")
        print(out[outcome].value_counts(normalize=True, dropna=False))

        print("\nGrouping levels:")
        for col in group_vars:
            print(f"{col}: {out[col].nunique()} levels")

    out["candidate"] = out["candidate"].astype(int)
    out["window_idx"] = pd.Categorical(
        out["window_idx"],
        categories=out["window_idx"].dropna().unique(),
    )
    out["clade"] = pd.Categorical(
        out["clade"],
        categories=out["clade"].dropna().unique(),
    )
    out["cluster_id"] = pd.Categorical(
        out["cluster_id"],
        categories=out["cluster_id"].dropna().unique(),
    )

    return out


def fit_bayesian_logistic_model(
    data: pd.DataFrame,
    formula: str,
    *,
    family: str = "bernoulli",
    categorical: list[str] | None = None,
    fixed_prior_sigma: float = 1.0,
    intercept_prior_sigma: float = 1.5,
    random_effect_sigma: float = 1.0,
    draws: int = 2000,
    tune: int = 2000,
    chains: int = 4,
    cores: int = 4,
    target_accept: float = 0.99,
    random_seed: int = 123,
    log_likelihood: bool = True,
    noncentered: bool = True,
):
    """
    Fit a Bayesian hierarchical logistic regression model using Bambi.

    Parameters
    ----------
    data : pandas.DataFrame
        Model dataframe.
    formula : str
        Bambi model formula, e.g.
        "candidate ~ age_entropy_z + (1|window_idx) + (1|clade)".
    family : str
        Model family. Defaults to "bernoulli".
    categorical : list[str] or None
        Variables to treat as categorical. If None, random-effect grouping
        variables are automatically treated as categorical.
    fixed_prior_sigma : float
        Standard deviation for fixed-effect Normal(0, sigma) priors.
    intercept_prior_sigma : float
        Standard deviation for the intercept prior.
    random_effect_sigma : float
        Scale for the HalfNormal prior on random-effect standard deviations.
    draws, tune, chains, cores, target_accept, random_seed
        Sampling arguments passed to model.fit().
    log_likelihood : bool
        Whether to store pointwise log-likelihood in the returned InferenceData.
    noncentered : bool
        Whether to use non-centred parameterisation.

    Returns
    -------
    model : bambi.Model
        Fitted Bambi model object.
    idata : arviz.InferenceData
        Posterior inference data.
    """

    # ------------------------------------------------------------
    # Extract response variable
    # ------------------------------------------------------------
    if "~" not in formula:
        raise ValueError("Formula must contain '~', e.g. 'candidate ~ x + (1|group)'.")

    response = formula.split("~")[0].strip()

    if response not in data.columns:
        raise ValueError(f"Response variable '{response}' not found in data.")

    # ------------------------------------------------------------
    # Intercept prior centred on observed outcome prevalence
    # ------------------------------------------------------------
    outcome_mean = data[response].mean()

    # Avoid infinite logit if the outcome is all 0s or all 1s
    outcome_mean = np.clip(outcome_mean, 1e-6, 1 - 1e-6)

    priors = {
        "Intercept": bmb.Prior(
            "Normal",
            mu=logit_func(outcome_mean),
            sigma=intercept_prior_sigma,
        )
    }

    # ------------------------------------------------------------
    # Extract random-effect grouping variables from terms like (1|clade)
    # ------------------------------------------------------------
    random_effect_matches = re.findall(r"\([^|()]+\|([^()]+)\)", formula)
    random_effect_groups = [x.strip() for x in random_effect_matches]

    for group in random_effect_groups:
        priors[f"1|{group}"] = bmb.Prior(
            "Normal",
            mu=0,
            sigma=bmb.Prior("HalfNormal", sigma=random_effect_sigma),
        )

    # ------------------------------------------------------------
    # Extract simple fixed-effect terms from the RHS
    # ------------------------------------------------------------
    rhs = formula.split("~", 1)[1]

    # Remove random-effect terms before parsing fixed effects
    rhs_without_random = re.sub(r"\([^()]*\|[^()]*\)", "", rhs)

    fixed_terms = [
        term.strip()
        for term in rhs_without_random.split("+")
        if term.strip() not in {"", "1", "0", "-1"}
    ]

    for term in fixed_terms:
        priors[term] = bmb.Prior(
            "Normal",
            mu=0,
            sigma=fixed_prior_sigma,
        )

    # ------------------------------------------------------------
    # Treat random-effect grouping variables as categorical by default
    # ------------------------------------------------------------
    if categorical is None:
        categorical = random_effect_groups

    # ------------------------------------------------------------
    # Build and fit model
    # ------------------------------------------------------------
    model = bmb.Model(
        formula=formula,
        data=data,
        family=family,
        priors=priors,
        categorical=categorical,
        noncentered=noncentered,
    )

    idata = model.fit(
        draws=draws,
        tune=tune,
        chains=chains,
        cores=cores,
        target_accept=target_accept,
        random_seed=random_seed,
        idata_kwargs={"log_likelihood": log_likelihood},
    )

    return model, idata


def _print_section(title: str, char: str = "=") -> None:
    """Print a clean section heading."""
    print(f"\n{title}")
    print(char * len(title))


def _format_df_for_print(
    df: pd.DataFrame,
    *,
    float_digits: int = 4,
    width: int = 160,
    max_colwidth: int = 80,
) -> str:
    """Return a neatly formatted DataFrame string for console printing."""
    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.width",
        width,
        "display.max_colwidth",
        max_colwidth,
        "display.float_format",
        lambda x: f"{x:,.{float_digits}f}",
    ):
        return df.to_string()


def _show_table(
    df: pd.DataFrame,
    *,
    display_tables: bool = False,
    float_digits: int = 4,
) -> None:
    """
    Display as a styled table in notebooks if requested;
    otherwise print a neatly aligned plain-text table.
    """
    if display_tables:
        try:
            from IPython.display import display

            display(df.style.format(precision=float_digits))
            return
        except Exception:
            pass

    print(_format_df_for_print(df, float_digits=float_digits))


def summarise_bambi_idata(
    idata: az.InferenceData,
    *,
    var_names=None,
    hdi_prob=0.95,
    odds_ratio_vars=None,
    print_diagnostics=True,
    rhat_threshold=1.01,
    ess_threshold=400,
    display_tables=False,
    float_digits=4,
) -> pd.DataFrame:
    """
    Print formatted Bayesian sampling diagnostics and return an ArviZ summary dataframe.

    Parameters
    ----------
    idata : arviz.InferenceData
        Posterior object returned by Bambi/PyMC.
    var_names : list[str] or None
        Variables to summarise. If None, summarises all posterior variables.
    hdi_prob : float
        HDI probability, e.g. 0.95 for 95% credible intervals.
    odds_ratio_vars : list[str] or None
        Variables to also summarise on odds-ratio scale via exp(beta).
        Usually use this for fixed-effect log-odds coefficients.
    print_diagnostics : bool
        Whether to print sampler diagnostics.
    rhat_threshold : float
        Warning threshold for R-hat.
    ess_threshold : int
        Warning threshold for bulk/tail ESS.
    display_tables : bool
        If True, displays styled tables in Jupyter notebooks.
        If False, prints aligned console-friendly tables.
    float_digits : int
        Number of decimal places to use in printed summaries.

    Returns
    -------
    summary_df : pandas.DataFrame
        ArviZ posterior summary dataframe.
    """

    summary_df = az.summary(
        idata,
        var_names=var_names,
        hdi_prob=hdi_prob,
        round_to=float_digits,
    )

    # ------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------
    if print_diagnostics:
        _print_section("Bayesian model diagnostics")

        diagnostic_rows = []

        # Divergences
        if hasattr(idata, "sample_stats") and "diverging" in idata.sample_stats:
            n_div = int(idata.sample_stats["diverging"].sum().item())
            n_total = int(idata.sample_stats["diverging"].size)
            div_rate = n_div / n_total

            diagnostic_rows.append(
                {
                    "Diagnostic": "Divergences",
                    "Value": f"{n_div} / {n_total} ({div_rate:.2%})",
                    "Status": "OK" if n_div == 0 else "WARNING",
                    "Interpretation": (
                        "No divergent transitions."
                        if n_div == 0
                        else "Investigate divergent transitions."
                    ),
                }
            )
        else:
            diagnostic_rows.append(
                {
                    "Diagnostic": "Divergences",
                    "Value": "Not available",
                    "Status": "NA",
                    "Interpretation": "No diverging field found in idata.sample_stats.",
                }
            )

        # BFMI
        try:
            bfmi = np.asarray(az.bfmi(idata))
            min_bfmi = np.nanmin(bfmi)
            bfmi_by_chain = ", ".join(f"{x:.3f}" for x in bfmi)

            diagnostic_rows.append(
                {
                    "Diagnostic": "BFMI",
                    "Value": f"min={min_bfmi:.3f}; chains=[{bfmi_by_chain}]",
                    "Status": "OK" if min_bfmi >= 0.3 else "WARNING",
                    "Interpretation": (
                        "Energy exploration looks acceptable."
                        if min_bfmi >= 0.3
                        else "One or more chains have BFMI < 0.3."
                    ),
                }
            )
        except Exception as e:
            diagnostic_rows.append(
                {
                    "Diagnostic": "BFMI",
                    "Value": "Could not compute",
                    "Status": "NA",
                    "Interpretation": str(e),
                }
            )

        # R-hat
        if "r_hat" in summary_df.columns:
            max_rhat = summary_df["r_hat"].max(skipna=True)

            diagnostic_rows.append(
                {
                    "Diagnostic": "Max R-hat",
                    "Value": f"{max_rhat:.4f}",
                    "Status": "OK" if max_rhat <= rhat_threshold else "WARNING",
                    "Interpretation": (
                        f"All parameters are at or below {rhat_threshold}."
                        if max_rhat <= rhat_threshold
                        else f"Some parameters exceed {rhat_threshold}."
                    ),
                }
            )

        # Bulk ESS
        if "ess_bulk" in summary_df.columns:
            min_bulk_ess = summary_df["ess_bulk"].min(skipna=True)

            diagnostic_rows.append(
                {
                    "Diagnostic": "Min bulk ESS",
                    "Value": f"{min_bulk_ess:.1f}",
                    "Status": "OK" if min_bulk_ess >= ess_threshold else "WARNING",
                    "Interpretation": (
                        f"All bulk ESS values are at least {ess_threshold}."
                        if min_bulk_ess >= ess_threshold
                        else f"Some bulk ESS values are below {ess_threshold}."
                    ),
                }
            )

        # Tail ESS
        if "ess_tail" in summary_df.columns:
            min_tail_ess = summary_df["ess_tail"].min(skipna=True)

            diagnostic_rows.append(
                {
                    "Diagnostic": "Min tail ESS",
                    "Value": f"{min_tail_ess:.1f}",
                    "Status": "OK" if min_tail_ess >= ess_threshold else "WARNING",
                    "Interpretation": (
                        f"All tail ESS values are at least {ess_threshold}."
                        if min_tail_ess >= ess_threshold
                        else f"Some tail ESS values are below {ess_threshold}."
                    ),
                }
            )

        # Tree depth
        if hasattr(idata, "sample_stats") and "tree_depth" in idata.sample_stats:
            max_tree_depth = int(idata.sample_stats["tree_depth"].max().item())

            diagnostic_rows.append(
                {
                    "Diagnostic": "Max tree depth",
                    "Value": str(max_tree_depth),
                    "Status": "INFO",
                    "Interpretation": "Maximum observed tree depth.",
                }
            )

        diagnostics_df = pd.DataFrame(diagnostic_rows)
        _show_table(
            diagnostics_df,
            display_tables=display_tables,
            float_digits=float_digits,
        )

        _print_section("Posterior summary")
        _show_table(
            summary_df,
            display_tables=display_tables,
            float_digits=float_digits,
        )

    # ------------------------------------------------------------
    # Optional odds-ratio summaries
    # ------------------------------------------------------------
    if odds_ratio_vars is not None:
        _print_section("Odds-ratio summaries")

        for var in odds_ratio_vars:
            if var not in idata.posterior:
                print(f"\n{var}: not found in idata.posterior")
                continue

            beta = idata.posterior[var]
            odds_ratio = np.exp(beta)

            or_idata = odds_ratio.to_dataset(name=f"OR_{var}")
            or_summary = az.summary(
                or_idata,
                hdi_prob=hdi_prob,
                round_to=float_digits,
            )

            p_beta_gt_0 = float((beta > 0).mean().item())
            p_beta_lt_0 = float((beta < 0).mean().item())
            p_or_gt_1 = float((odds_ratio > 1).mean().item())
            p_or_lt_1 = float((odds_ratio < 1).mean().item())

            _print_section(var, char="-")

            _show_table(
                or_summary,
                display_tables=display_tables,
                float_digits=float_digits,
            )

            prob_df = pd.DataFrame(
                {
                    "Quantity": [
                        "P(beta > 0 | data)",
                        "P(beta < 0 | data)",
                        "P(OR > 1 | data)",
                        "P(OR < 1 | data)",
                    ],
                    "Probability": [
                        p_beta_gt_0,
                        p_beta_lt_0,
                        p_or_gt_1,
                        p_or_lt_1,
                    ],
                }
            )

            _show_table(
                prob_df,
                display_tables=display_tables,
                float_digits=float_digits,
            )

    return summary_df

## Composition models: sequence-level association

### Single model with primary adjusters

```text
candidate ~ C(<predictor>, Treatment(reference=<reference>))
          + C(window_idx)
          + C(clade)
```

### Joint model with primary adjusters

```text
candidate ~ C(sex, Treatment(reference='Male'))
          + C(age_band, Treatment(reference='20-24'))
          + C(dz_simd_quintile, Treatment(reference='1'))
          + C(dz_urban_rural_class, Treatment(reference='Large Urban Areas'))
          + C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))
          + C(window_idx)
          + C(clade)
```

### Additional adjusters

```text
+ z_dz_cum_prop_sequenced
+ z_dz_cum_incidence_per_capita
+ z_dz_7d_test_positivity
+ z_log1p_dz_cum_positive_tests
```

In [5]:
composition_predictors = {
    "sex": "Male",
    "age_band": "20-24",
    "dz_simd_quintile": "1",
    "dz_urban_rural_class": "Large Urban Areas",
    "dz_health_board": "Greater Glasgow and Clyde",
}

comp_model_df = get_complete_case_data(
    df=eligible_sequence_data,
    outcome="candidate",
    predictors=composition_predictors.keys(),
    group_vars=("window_idx", "clade"),
    categorical_vars=("window_idx", "clade"),
)

Complete-case summary
---------------------
Rows before dropna: 264,139
Rows after dropna:  264,139
Rows dropped:       0
Percent retained:   100.0%

Outcome counts:
candidate
0    197299
1     66840
Name: count, dtype: int64

Outcome proportions:
candidate
0    0.746951
1    0.253049
Name: proportion, dtype: float64

Grouping levels:
window_idx: 67 levels
clade: 21 levels


In [6]:
seq_subset = sample_model_test_data(
    comp_model_df,
    max_rows=1000,
    positive_fraction=candidate_sequence_rate,
    categorical_vars=list(composition_predictors.keys())
    + ["window_idx", "clade", "cluster_id"],
)

form = (
    "candidate ~ "
    "C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')) "
    "+ (1|window_idx) + (1|clade)"
)

priors = {
    "Intercept": bmb.Prior(
        "Normal",
        mu=logit_func(seq_subset["candidate"].mean()),
        sigma=1.5,
    ),
    "common": bmb.Prior("Normal", mu=0, sigma=1),
    "group_specific": bmb.Prior(
        "Normal",
        mu=0,
        sigma=bmb.Prior("HalfNormal", sigma=1),
    ),
}

model = bmb.Model(
    formula=form,
    data=seq_subset,
    family="bernoulli",
    priors=priors,
    categorical=["window_idx", "clade"],
    noncentered=True,
)

idata = model.fit(
    draws=2000,
    tune=2000,
    chains=4,
    cores=4,
    target_accept=0.99,
    random_seed=123,
    idata_kwargs={"log_likelihood": True},
)

Modeling the probability that candidate==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde')), 1|window_idx_sigma, 1|window_idx_offset, 1|clade_sigma, 1|clade_offset]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 36 seconds.


In [8]:
summary_df = summarise_bambi_idata(
    idata,
    var_names=[
            "Intercept",
            "C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))",
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        hdi_prob=0.95,
        odds_ratio_vars=["C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))"],
        display_tables=True,
    )


Bayesian model diagnostics


,Diagnostic,Value,Status,Interpretation
0,Divergences,0 / 8000 (0.00%),OK,No divergent transitions.
1,BFMI,"min=0.776; chains=[0.781, 0.885, 0.776, 0.841]",OK,Energy exploration looks acceptable.
2,Max R-hat,1.0019,OK,All parameters are at or below 1.01.
3,Min bulk ESS,1954.4,OK,All bulk ESS values are at least 400.
4,Min tail ESS,2014.9,OK,All tail ESS values are at least 400.
5,Max tree depth,8,INFO,Maximum observed tree depth.



Posterior summary


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,-1.9846,0.4430,-2.8943,-1.1999,0.0093,0.0073,2287.6740,3364.4522,1.0006
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Ayrshire and Arran]",0.0062,0.3108,-0.6062,0.5966,0.0040,0.0033,6047.9699,5833.7068,1.0002
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Borders]",0.4562,0.5428,-0.5609,1.5564,0.0066,0.0059,6859.5864,6444.9799,1.0003
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Dumfries and Galloway]",-0.5638,0.5424,-1.6320,0.4784,0.0058,0.0065,8821.2436,5394.0511,1.0002
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Fife]",0.3014,0.2990,-0.2825,0.8797,0.0037,0.0034,6375.1334,5726.8578,1.0000
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Forth Valley]",-0.0411,0.3354,-0.6750,0.6412,0.0042,0.0036,6394.6692,5507.7130,1.0004
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Grampian]",0.2607,0.2802,-0.3111,0.7852,0.0038,0.0030,5518.9413,5849.1518,1.0008
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Highland]",-0.7092,0.4268,-1.5978,0.0715,0.0051,0.0048,7148.8144,5871.7133,1.0002
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lanarkshire]",0.2482,0.2685,-0.2844,0.7738,0.0035,0.0029,5937.1460,5217.8559,1.0005
"C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lothian]",0.2866,0.2259,-0.1721,0.7102,0.0032,0.0023,4958.2518,5592.1517,1.0019



Odds-ratio summaries

C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))
--------------------------------------------------------------------


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Ayrshire and Arran]",1.0556,0.3317,0.5057,1.7308,0.0042,0.0043,6047.9699,5833.7068,1.0002
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Borders]",1.8263,1.0607,0.3271,3.8588,0.0129,0.0237,6859.5864,6444.9799,1.0003
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Dumfries and Galloway]",0.6565,0.3681,0.1135,1.3713,0.0039,0.0061,8821.2436,5394.0511,1.0002
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Fife]",1.4134,0.4322,0.6627,2.2597,0.0055,0.0062,6375.1334,5726.8578,1.0000
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Forth Valley]",1.0150,0.3470,0.4174,1.6949,0.0044,0.0045,6394.6692,5507.7130,1.0004
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Grampian]",1.3496,0.3835,0.6713,2.1057,0.0051,0.0049,5518.9413,5849.1518,1.0008
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Highland]",0.5376,0.2308,0.1624,1.0027,0.0027,0.0034,7148.8144,5871.7133,1.0002
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lanarkshire]",1.3288,0.3633,0.6692,2.0389,0.0048,0.0049,5937.1460,5217.8559,1.0004
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Lothian]",1.3664,0.3129,0.8033,1.9774,0.0045,0.0039,4958.2518,5592.1517,1.0022
"OR_C(dz_health_board, Treatment(reference='Greater Glasgow and Clyde'))[Orkney]",2.0356,2.0224,0.0797,5.6300,0.0230,0.0575,8758.7381,6333.6001,1.0002


,Quantity,Probability
0,P(beta > 0 | data),0.6018
1,P(beta < 0 | data),0.3982
2,P(OR > 1 | data),0.6018
3,P(OR < 1 | data),0.3982


## Mixing models: node-level association

### Single model with primary adjusters

```text
candidate ~ <predictor> + C(window_idx) + C(clade)
```

### Joint model with primary adjusters

```text
candidate ~ sex_entropy_z
          + age_entropy_z
          + simd_entropy_z
          + datazone_entropy_z
          + local_authority_entropy_z
          + urban_rural_entropy_z
          + health_board_entropy_z
          + vaccination_entropy_z
          + C(window_idx)
          + C(clade)
```

### Additional adjusters

```text
+ z_dz_cum_prop_sequenced
+ z_dz_cum_incidence_per_capita
+ z_dz_7d_test_positivity
+ z_log1p_dz_cum_positive_tests
```

In [ ]:
mixing_predictors = [
    "sex_entropy_z",
    "age_entropy_z",
    "simd_entropy_z",
    "datazone_entropy_z",
    "local_authority_entropy_z",
    "urban_rural_entropy_z",
    "health_board_entropy_z",
    "vaccination_entropy_z",
]

model_df = get_complete_case_data(
    df=eligible_nodes,
    outcome="candidate",
    predictors=mixing_predictors,
    group_vars=("window_idx", "clade"),
    categorical_vars=("window_idx", "clade"),
)

In [ ]:
single_primary_results = {}

for col in mixing_predictors:
    if col not in model_df.columns:
        raise ValueError(f"Predictor column '{col}' not found in model dataframe.")

    form = f"candidate ~ {col} + (1|window_idx) + (1|clade)"

    model, idata = fit_bayesian_logistic_model(
        data=model_df,
        formula=form,
    )
    summary = summarise_bambi_idata(
        idata,
        var_names=[
            "Intercept",
            f"{col}",
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        hdi_prob=0.95,
        odds_ratio_vars=[f"{col}"],
        display_tables=True,
    )

    single_primary_results[col] = {"model": model, "idata": idata, "summary": summary}

In [ ]:
single_expanded_results = {}

for col in mixing_predictors:
    if col not in model_df.columns:
        raise ValueError(f"Predictor column '{col}' not found in model dataframe.")

    form = f"candidate ~ {col} + (1|window_idx) + (1|clade) + z_dz_cum_prop_sequenced + z_dz_cum_incidence_per_capita + z_dz_7d_test_positivity + z_log1p_dz_cum_positive_tests"

    model, idata = fit_bayesian_logistic_model(
        data=model_df,
        formula=form,
    )
    summary = summarise_bambi_idata(
        idata,
        var_names=[
            "Intercept",
            f"{col}",
            "1|window_idx_sigma",
            "1|clade_sigma",
        ],
        hdi_prob=0.95,
        odds_ratio_vars=[f"{col}"],
        display_tables=True,
    )

    single_expanded_results[col] = {"model": model, "idata": idata, "summary": summary}

In [ ]:
joint_primary_results = {}

predictors = [col for col in mixing_predictors if col in model_df.columns]

form = f"candidate ~ {' + '.join(predictors)} + (1|window_idx) + (1|clade)"

model, idata = fit_bayesian_logistic_model(
    data=model_df,
    formula=form,
)
summary = summarise_bambi_idata(
    idata,
    var_names=[
        "Intercept",
        "1|window_idx_sigma",
        "1|clade_sigma",
    ]
    + predictors,
    hdi_prob=0.95,
    odds_ratio_vars=[f"{col}"],
    display_tables=True,
)

joint_primary_results["joint_primary"] = {"model": model, "idata": idata, "summary": summary}

In [ ]:
joint_expanded_results = {}

predictors = [col for col in mixing_predictors if col in model_df.columns]

form = f"candidate ~ {' + '.join(predictors)} + (1|window_idx) + (1|clade) + z_dz_cum_prop_sequenced + z_dz_cum_incidence_per_capita + z_dz_7d_test_positivity + z_log1p_dz_cum_positive_tests"

model, idata = fit_bayesian_logistic_model(
    data=model_df,
    formula=form,
)
summary = summarise_bambi_idata(
    idata,
    var_names=[
        "Intercept",
        "1|window_idx_sigma",
        "1|clade_sigma",
    ]
    + predictors,
    hdi_prob=0.95,
    odds_ratio_vars=[f"{col}"],
    display_tables=True,
)

joint_expanded_results["joint_expanded"] = {"model": model, "idata": idata, "summary": summary}